# Evaluate HelpSteer2 M1 and C1 Adapter Merges

This notebook evaluates five-objective GPT-2 LoRA adapter merges using four coefficient-selection methods:

- uniform coefficients,
- direct-preference coefficients,
- M1 relationship-softmax coefficients,
- C1 CAGrad-inspired coefficients.

For each setting, the evaluation generates responses for four shared prompts, computes lightweight HelpSteer2 proxy scores, aggregates preference-weighted utilities, and writes comparison files under `results/`.

## Check the GPU

In Colab, select **Runtime > Change runtime type > T4 GPU** before starting. The evaluation repeatedly loads GPT-2 and five LoRA adapters, so a GPU is strongly recommended.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Generation on CPU will be considerably slower.")

## Clone or update the repository

The following cell starts from `/content`. It updates an existing valid repository or clones a fresh copy, avoiding nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")

## Show the repository structure

Confirm that the evaluation script, coefficient file, and results folder are available.

In [ ]:
!pwd
!ls
!ls scripts
!ls results

## Install dependencies

The setup removes an old preinstalled `torchao` version because it can conflict with recent Transformers releases. The evaluation uses Transformers, PEFT, Accelerate, pandas, NumPy, and SciPy.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U transformers datasets peft accelerate safetensors pandas numpy scipy

If Colab reports that a package was already imported, restart the runtime and begin again from the first cell. Install a current `torchao` version with `!pip install -q -U "torchao>=0.16.0"` only when another package explicitly requires it.

## Check the coefficient file

Notebook 12 creates `results/helpsteer2_m1_c1_coefficients.csv`. The evaluation reads its direct-preference, M1, and C1 lambda vectors and adds a uniform baseline for every preference.

In [ ]:
from pathlib import Path

coefficient_path = Path("results/helpsteer2_m1_c1_coefficients.csv")

if not coefficient_path.is_file():
    raise FileNotFoundError(
        f"Missing coefficient file: {coefficient_path}. "
        "Run Notebook 12 or scripts/compute_helpsteer2_m1_c1_coefficients.py first."
    )

print(f"Coefficient file found: {coefficient_path}")

In [ ]:
import pandas as pd

coefficient_df = pd.read_csv(coefficient_path)
print(f"Coefficient rows: {len(coefficient_df)}")
coefficient_df.head()

## Check the local HelpSteer2 adapters

All five objective-specific adapter folders are required. If every folder is marked `FOUND`, skip the upload section.

In [ ]:
adapter_paths = [
    Path("adapters/helpsteer2-gpt2-helpfulness-adapter"),
    Path("adapters/helpsteer2-gpt2-correctness-adapter"),
    Path("adapters/helpsteer2-gpt2-coherence-adapter"),
    Path("adapters/helpsteer2-gpt2-complexity-adapter"),
    Path("adapters/helpsteer2-gpt2-verbosity-adapter"),
]

for path in adapter_paths:
    status = "FOUND" if path.is_dir() else "MISSING"
    print(f"{status:7} {path}")

print(f"\nAll adapters available: {all(path.is_dir() for path in adapter_paths)}")

## Upload the adapter backup if needed

Run the next two cells when one or more adapter folders are missing. Select your local `helpsteer2_adapters.zip` backup.

**Important:** The ZIP is a local backup of generated adapter weights. Neither the ZIP nor the extracted `adapters/` folder belongs in GitHub.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!if [ -f helpsteer2_adapters.zip ]; then unzip -o helpsteer2_adapters.zip; else echo "No helpsteer2_adapters.zip found, skipping unzip."; fi
!ls adapters || echo "No adapters folder found."

## Verify the adapter files

The checker confirms that all five adapter folders contain the expected PEFT configuration and weight files.

In [ ]:
!python scripts/check_helpsteer2_adapters.py

## Evaluate the M1 and C1 adapter merges

The script evaluates 32 preference-and-method settings across four prompts. It caches identical lambda vectors, but this generation step can still take several minutes because each unique merge loads GPT-2 and five LoRA adapters.

In [ ]:
!python scripts/evaluate_helpsteer2_m1_c1_merges.py

## Inspect the output files

The evaluation creates:

- `results/helpsteer2_m1_c1_merge_generations.csv`
- `results/helpsteer2_m1_c1_scored_generations.csv`
- `results/helpsteer2_m1_c1_comparison.csv`
- `results/helpsteer2_m1_c1_comparison_metadata.json`

In [ ]:
!ls results

### Generated responses

This file contains one row per preference, method setting, and prompt.

In [ ]:
generations_df = pd.read_csv(
    "results/helpsteer2_m1_c1_merge_generations.csv"
)
print(f"Rows: {len(generations_df)}")
display(generations_df.head())

### Scored responses

This file adds five lightweight proxy scores, response length, and an empty-response flag to every generation.

In [ ]:
scored_df = pd.read_csv(
    "results/helpsteer2_m1_c1_scored_generations.csv"
)
print(f"Rows: {len(scored_df)}")
display(scored_df.head())

### Aggregated comparison

This table summarizes the mean proxy scores and preference-weighted utility for every method setting.

In [ ]:
comparison_df = pd.read_csv(
    "results/helpsteer2_m1_c1_comparison.csv"
)
print(f"Comparison rows: {len(comparison_df)}")
display(comparison_df)

### Evaluation metadata

The metadata records input and output paths, objective order, evaluated methods, prompt count, and generation settings.

In [ ]:
import json

metadata_path = Path(
    "results/helpsteer2_m1_c1_comparison_metadata.json"
)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
print(json.dumps(metadata, indent=2))

## Understand the comparison columns

- `preference_name` identifies the user preference vector.
- `method` identifies `uniform`, `direct_preference`, `M1`, or `C1`.
- `hyperparameter_name` and `hyperparameter_value` record $\tau$ for M1 or $\rho$ for C1.
- `lambda_*` columns contain the five merge coefficients.
- `mean_*_proxy` columns contain the average lightweight attribute scores across the four prompts.
- `utility_for_preference` weights the mean proxy scores by the original preference vector.
- `l1_distance_to_p` and `l2_distance_to_p` measure coefficient movement from the preference vector.
- `gap_to_best_fixed_sweep` compares the method utility with the best tested fixed-sweep utility when that summary is available.

The proxy scores are infrastructure-level heuristics rather than reward-model scores or human HelpSteer2 labels.

## Git safety check

The small CSV and JSON result files are suitable for version control.

Do not commit:

- `adapters/`
- ZIP backup files
- `.safetensors` or `.bin` files
- checkpoints or full model files

In [ ]:
!git status